In [ ]:
import os
import glob
import json
import numpy as np
import torch
import h5py
from pathlib import Path
from typing import Tuple, Dict, Optional

# --------- USER CONFIG ---------
PT_DIR   = r"C:\path\to\per_slide_pt"      # one .pt per slide (features + coords)
NPZ_DIR  = r"C:\path\to\per_slide_npz"     # one .npz per slide (qq + coords or qq only)
OUT_DIR  = r"C:\path\to\merged_npz"
SLIDE_ID_FROM_NAME = True                   # if True, slide_id = stem of filename
COORD_TOL = 0                               # 0 for exact integer match; try 1 if off-by-1s appear
# Optional: CSV with slide-level labels if you want a single concatenated training set
CSV_LABELS = None  # e.g., r"C:\...\train_val_test_labels.csv" with columns: slide_id,label  (label in {FA,PT})

LABEL_MAP = {"FA":0, "PT":1}  # adjust as needed
# --------------------------------

def load_pt_or_h5(fp: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns (features[N,d], coords[N,2]) from a .pt or .h5 file.
    Expected keys (common patterns): 'features'/'feats', 'coords'/'xy'/'patch_coords'
    """
    p = Path(fp)
    if p.suffix.lower() == ".pt":
        obj = torch.load(fp, map_location="cpu")
        # Try common key names
        feats = None
        coords = None
        for k in ["features", "feats", "emb", "Z", "z"]:
            if k in obj:
                feats = obj[k]
                break
        for k in ["coords", "xy", "patch_coords", "xy_top_left"]:
            if k in obj:
                coords = obj[k]
                break
        if isinstance(feats, torch.Tensor): feats = feats.numpy()
        if isinstance(coords, torch.Tensor): coords = coords.numpy()
        assert feats is not None and coords is not None, f"Missing 'features'/'coords' in {fp}. Keys: {list(obj.keys())}"
        feats = np.asarray(feats)
        coords = np.asarray(coords)
        return feats, coords

    elif p.suffix.lower() == ".h5" or p.suffix.lower() == ".hdf5":
        with h5py.File(fp, "r") as f:
            # Try common dataset names
            def first_key(f, candidates):
                for c in candidates:
                    if c in f: return c
                return None
            kf = first_key(f, ["features", "feats", "Z", "z"])
            kc = first_key(f, ["coords", "xy", "patch_coords", "xy_top_left"])
            assert kf and kc, f"Missing datasets in {fp}. Have: {list(f.keys())}"
            feats = np.array(f[kf])
            coords = np.array(f[kc])
            return feats, coords
    else:
        raise ValueError(f"Unsupported feature file format: {fp}")

def load_npz_assignments(fp: str) -> Dict[str, np.ndarray]:
    """
    Returns dict with at least 'qq' and optional 'coords'.
    Your NPZ often contains: qq (N,K), coords (N,2), maybe mask/slide_id.
    """
    d = dict(np.load(fp, allow_pickle=True))
    # Normalize keys
    key_q = None
    for k in ["qq", "Q", "resp", "responsibilities", "assignments"]:
        if k in d:
            key_q = k
            break
    assert key_q is not None, f"No responsibilities found in {fp}. Keys: {list(d.keys())}"
    out = {"qq": np.asarray(d[key_q])}
    # coords (if present) to enable safe join when orders differ
    for k in ["coords", "xy", "patch_coords", "xy_top_left"]:
        if k in d:
            out["coords"] = np.asarray(d[k])
            break
    # passthrough others if you want them later (e.g., mask)
    return out

def make_index_from_coords(coords: np.ndarray, tol: int = 0) -> Dict[Tuple[int,int], int]:
    """
    Build a { (x,y) -> row_index } index. If tol>0, we expand keys in a small +/- tol neighborhood.
    This is simple and fast for integer coords; for very large N with tol>0 consider KDTree.
    """
    coords = np.asarray(coords, dtype=np.int64)
    idx = {}
    if tol == 0:
        for i, (x, y) in enumerate(coords):
            idx[(int(x), int(y))] = i
        return idx
    else:
        # For tolerant matching, create multiple keys per row
        # (works if tol is small like 1; for larger tol, use KDTree)
        offsets = [(dx, dy) for dx in range(-tol, tol+1) for dy in range(-tol, tol+1)]
        for i, (x, y) in enumerate(coords):
            xi, yi = int(x), int(y)
            for dx, dy in offsets:
                key = (xi+dx, yi+dy)
                # keep first occurrence; warn if collisions become an issue
                if key not in idx:
                    idx[key] = i
        return idx

def align_by_coords(coords_A: np.ndarray, coords_B: np.ndarray, tol: int = 0):
    """
    Align rows in A (features) to rows in B (responsibilities) using integer coords with tolerance.
    Returns (idxA, idxB) for matched pairs, and lists of unmatched indices for each side.
    """
    idxB = make_index_from_coords(coords_B, tol=tol)
    idxA_match = []
    idxB_match = []
    unmatched_A = []
    for i, (x, y) in enumerate(coords_A.astype(int)):
        hit = idxB.get((int(x), int(y)))
        if hit is None:
            unmatched_A.append(i)
        else:
            idxA_match.append(i)
            idxB_match.append(hit)
    unmatched_B = sorted(set(range(len(coords_B))) - set(idxB_match))
    return np.array(idxA_match), np.array(idxB_match), np.array(unmatched_A), np.array(unmatched_B)

def merge_one_slide(pt_fp: str, npz_fp: str, slide_id: Optional[str] = None, tol: int = 0) -> Dict[str, np.ndarray]:
    Z, C_feat = load_pt_or_h5(pt_fp)           # (Nf, d), (Nf, 2)
    npz = load_npz_assignments(npz_fp)         # qq: (Na, K); coords? (Na,2) maybe

    if slide_id is None:
        slide_id = Path(pt_fp).stem

    if "coords" in npz:
        C_assign = npz["coords"]
        iA, iB, unA, unB = align_by_coords(C_feat, C_assign, tol=tol)
        if len(iA) == 0:
            raise RuntimeError(f"No coordinate matches between {pt_fp} and {npz_fp} (tol={tol}).")
        Zm = Z[iA]
        Cm = C_feat[iA]
        Qm = npz["qq"][iB]
        if len(unA) or len(unB):
            print(f"[{slide_id}] matched {len(iA)} rows. Unmatched: features={len(unA)}, qq={len(unB)} (tol={tol})")
    else:
        # Fallback: assume same order/length
        assert len(Z) == len(npz["qq"]), \
            f"Count mismatch for {slide_id}: features={len(Z)} vs qq={len(npz['qq'])}, and npz has no coords."
        Zm = Z
        Cm = C_feat
        Qm = npz["qq"]
        print(f"[{slide_id}] merged by order: N={len(Zm)}")

    out = {
        "slide_id": np.array([slide_id]*len(Zm), dtype=object),
        "Z": Zm.astype(np.float32),
        "Q": Qm.astype(np.float32),
        "coords": Cm.astype(np.int64),
    }
    return out

def main_build_per_slide(PT_DIR, NPZ_DIR, OUT_DIR, tol=COORD_TOL):
    os.makedirs(OUT_DIR, exist_ok=True)
    # Map slides by stem
    pt_files = {Path(f).stem: f for f in glob.glob(os.path.join(PT_DIR, "*.pt"))}
    # optionally include h5
    for f in glob.glob(os.path.join(PT_DIR, "*.h5")) + glob.glob(os.path.join(PT_DIR, "*.hdf5")):
        pt_files[Path(f).stem] = f

    npz_files = {Path(f).stem: f for f in glob.glob(os.path.join(NPZ_DIR, "*.npz"))}

    common = sorted(set(pt_files).intersection(npz_files))
    missing_pt = sorted(set(npz_files) - set(pt_files))
    missing_npz = sorted(set(pt_files) - set(npz_files))

    if missing_pt:
        print(f"Slides missing .pt: {len(missing_pt)} → {missing_pt[:5]}...")
    if missing_npz:
        print(f"Slides missing .npz: {len(missing_npz)} → {missing_npz[:5]}...")

    merged_index = []
    for sid in common:
        try:
            merged = merge_one_slide(pt_files[sid], npz_files[sid], slide_id=sid, tol=tol)
            out_fp = os.path.join(OUT_DIR, f"{sid}.npz")
            np.savez_compressed(out_fp, **merged)
            merged_index.append({"slide_id": sid, "path": out_fp, "N": len(merged["Z"])})
        except Exception as e:
            print(f"[{sid}] ERROR: {e}")

    # Save a small manifest for convenience
    with open(os.path.join(OUT_DIR, "_manifest.json"), "w") as fh:
        json.dump(merged_index, fh, indent=2)
    print(f"Done. Wrote {len(merged_index)} merged slides to {OUT_DIR}")

def optional_concat_dataset(OUT_DIR, CSV_LABELS=None, LABEL_MAP=None):
    """
    If you want a single (X) dataset across slides for patch-level training:
    returns (Z_all, Q_all, coords_all, y_all, slide_ids)
    """
    # load manifest
    man_fp = os.path.join(OUT_DIR, "_manifest.json")
    assert os.path.exists(man_fp), "Run main_build_per_slide first."
    manifest = json.load(open(man_fp, "r"))

    # Optional labels
    slide2y = None
    if CSV_LABELS:
        import pandas as pd
        df = pd.read_csv(CSV_LABELS)
        assert "slide_id" in df.columns and "label" in df.columns
        slide2y = {str(r.slide_id): LABEL_MAP[str(r.label)] for _, r in df.iterrows()}

    Zs, Qs, Cs, ys, sids = [], [], [], [], []
    for rec in manifest:
        d = dict(np.load(rec["path"], allow_pickle=True))
        Zs.append(np.asarray(d["Z"]))
        Qs.append(np.asarray(d["Q"]))
        Cs.append(np.asarray(d["coords"]))
        sid_arr = np.asarray(d["slide_id"]).astype(str)
        sids.append(sid_arr)
        if slide2y is not None:
            ys.append(np.full(len(sid_arr), slide2y[str(sid_arr[0])], dtype=np.int64))

    Z_all = np.concatenate(Zs, axis=0)
    Q_all = np.concatenate(Qs, axis=0)
    C_all = np.concatenate(Cs, axis=0)
    SID_all = np.concatenate(sids, axis=0)
    y_all = np.concatenate(ys, axis=0) if ys else None

    print(f"Concatenated: Z={Z_all.shape}, Q={Q_all.shape}, coords={C_all.shape}, slides={len(np.unique(SID_all))}")
    if y_all is not None:
        class_counts = {int(c): int((y_all==c).sum()) for c in np.unique(y_all)}
        print(f"Label counts: {class_counts}")
    return Z_all, Q_all, C_all, y_all, SID_all

if __name__ == "__main__":
    main_build_per_slide(PT_DIR, NPZ_DIR, OUT_DIR, tol=COORD_TOL)
    # Z_all, Q_all, C_all, y_all, SID_all = optional_concat_dataset(OUT_DIR, CSV_LABELS, LABEL_MAP)
